### **Parte Zero - ATENÇÃO:**
O seguinte algoritmo de de recomendação por similaridade do cosseno foi baseado nos códigos do repositório abaixo:
- https://github.com/Megalonnix/ProjSimilaridadeCossenoV2/tree/master/notebooks/definitivo
- Tal reaproveitamento veio da ideia de um dos membros do grupo considerar a remoção de stopwords.
- Tirando a habilidade acima, e a organização do código em pequenos módulos/funções individuais, todo o código a seguir usa propriedades nativas do R. 

In [2]:
if(!require(pacman)) install.packages(pacman)

Carregando pacotes exigidos: pacman



### **Parte 1A - Módulo de Tokenização + Remoção de Stopwords:**

In [3]:
pacman::p_load(stopwords)

# Os códigos abaixo giram em torno
# da tokenização de strings e a remoção
# de stopwords.

remove_punctuation <- function(txt) {
  txt <- tolower(txt)
  txt <- gsub("[[:punct:]]", "", txt)
  return(txt)
}

tokenize_txt <- function(txt) {
  words_from_txt <- unlist(strsplit(txt, "\\s+"))
  words_from_txt <- words_from_txt[words_from_txt != ""]
  return(words_from_txt)
}

remove_stopwords <- function(txt, desligado = FALSE) {
  token_from_txt <- tokenize_txt(remove_punctuation(txt))
  
  stopwords_set_1 <- stopwords::stopwords(language = "en", source = "snowball")
  stopwords_set_2 <- stopwords::stopwords(language = "pt", source = "snowball")
  stopwords_multilingual <- unique(c(stopwords_set_1, stopwords_set_2))
  
  if (desligado == FALSE) {
    token_without_stopwords <- token_from_txt[!token_from_txt %in% stopwords_multilingual]
    return(token_without_stopwords)
  } else {
    return(token_from_txt)
  }
}

### **Parte 1B - Testando Módulo de Tokenização + Remoção de Stopwords:**

In [4]:
sample_txts = c(
    'O gato pulou na mesa.',
    'The quick brown fox jumps over the lazy dog.'   
)

In [5]:
tokenize_txt(sample_txts[1])
tokenize_txt(sample_txts[2])

[1] "O"     "gato"  "pulou" "na"    "mesa."

[1] "The"   "quick" "brown" "fox"   "jumps" "over"  "the"   "lazy"  "dog."

In [6]:
remove_stopwords(txt = sample_txts[1],desligado = TRUE)
remove_stopwords(txt = sample_txts[1],desligado = FALSE)

[1] "o"     "gato"  "pulou" "na"    "mesa"

[1] "gato"  "pulou" "mesa"

In [7]:
remove_stopwords(txt = sample_txts[2],desligado = TRUE)
remove_stopwords(txt = sample_txts[2],desligado = FALSE)

[1] "the"   "quick" "brown" "fox"   "jumps" "over"  "the"   "lazy"  "dog"

[1] "quick" "brown" "fox"   "jumps" "lazy"  "dog"

### **Parte 2A - Módulo de criação das marizes Bag of Words (BoW) e TF-IDF:**

In [8]:
# Os códigos abaixo giram em torno
# de manusear textos já preprocessados,
# e fazer coisas como: (1) matriz BoW, (2) matriz TF-IDF.
#
# Diferente da primeira versão, aqui trabalhamos DIRETO com os tokens
# (sem reconstruir string e tokenizar de novo), já que não dependemos
# de nenhum vectorizer externo.

get_processed_tokens <- function(nonProcessedTexts,
                                  dontRemoveStopWords = FALSE) {
  lapply(nonProcessedTexts, function(txt) {
    remove_stopwords(txt, desligado = dontRemoveStopWords)
  })
}

get_BOW_matrix <- function(nonProcessedTexts,
                            dontRemoveStopWords = FALSE) {

  tokenized_corpus <- get_processed_tokens(nonProcessedTexts, dontRemoveStopWords)

  # vocabulário: todas as palavras únicas em todos os documentos
  vocabulary <- sort(unique(unlist(tokenized_corpus)))

  # nomes das colunas: recriamos as frases só para rotular a matriz
  # (não são usadas em nenhum processamento, é só exibição)
  doc_labels <- sapply(tokenized_corpus, paste, collapse = " ")

  bow_matrix <- matrix(0,
                        nrow = length(vocabulary),
                        ncol = length(tokenized_corpus),
                        dimnames = list(vocabulary, doc_labels))

  for (i in seq_along(tokenized_corpus)) {
    counts <- table(tokenized_corpus[[i]])
    bow_matrix[names(counts), i] <- as.numeric(counts)
  }

  df_bow_matrix <- as.data.frame(bow_matrix)

  return(list(matrix = bow_matrix, df = df_bow_matrix))
}

get_TFIDF_matrix <- function(nonProcessedTexts,
                              dontRemoveStopWords = FALSE) {

  tokenized_corpus <- get_processed_tokens(nonProcessedTexts, dontRemoveStopWords)

  vocabulary <- sort(unique(unlist(tokenized_corpus)))
  doc_labels <- sapply(tokenized_corpus, paste, collapse = " ")

  n_docs <- length(tokenized_corpus)

  # matriz de contagem (mesma lógica do BOW)
  count_matrix <- matrix(0,
                          nrow = length(vocabulary),
                          ncol = n_docs,
                          dimnames = list(vocabulary, doc_labels))

  for (i in seq_along(tokenized_corpus)) {
    counts <- table(tokenized_corpus[[i]])
    count_matrix[names(counts), i] <- as.numeric(counts)
  }

  # document frequency por termo
  doc_freq <- rowSums(count_matrix > 0)

  # idf suavizado (equivalente ao default do sklearn: smooth_idf=True)
  idf_vec <- log((1 + n_docs) / (1 + doc_freq)) + 1

  # tf-idf bruto (contagem x idf)
  tfidf_matrix <- count_matrix * idf_vec

  # normalização L2 por documento (coluna)
  col_norms <- sqrt(colSums(tfidf_matrix^2))
  col_norms[col_norms == 0] <- 1
  tfidf_matrix <- sweep(tfidf_matrix, 2, col_norms, "/")

  df_tfidf_matrix <- as.data.frame(tfidf_matrix)

  return(list(matrix = tfidf_matrix, df = df_tfidf_matrix))
}

### **Parte 2B - Testando Módulo de criação da matriz BOW + a matriz TF-IDF:**

In [9]:
textos_simples <- list("I love you", "Love")
textos_simples

[[1]]
[1] "I love you"

[[2]]
[1] "Love"

#### **Sub-teste: remoção de stopwords desativada:**

In [10]:
tokens <- get_processed_tokens(
    textos_simples, 
    dontRemoveStopWords = TRUE)
print(tokens)

[[1]]
[1] "i"    "love" "you" 

[[2]]
[1] "love"



In [11]:
resultado_bow <- get_BOW_matrix(
    textos_simples, 
    dontRemoveStopWords = TRUE)

# print(resultado_bow$matrix)
# print(resultado_bow$df)
resultado_bow$matrix

,i love you,love
i,1,0
love,1,1
you,1,0


In [12]:
resultado_tfidf <- get_TFIDF_matrix(
    textos_simples, 
    dontRemoveStopWords = TRUE)
# print(resultado_tfidf$matrix)
# print(resultado_tfidf$df)
resultado_tfidf$matrix

,i love you,love
i,0.6316672,0
love,0.4494364,1
you,0.6316672,0


##### **Explicação do código acima (Remoção de stopwords DESATIVADA):**

---

**Documento 1: "I love you"** $\rightarrow$ tokens: $[\text{i},\ \text{love},\ \text{you}]$

**Documento 2: "love"** $\rightarrow$ tokens: $[\text{love}]$

*(sem remoção de stopwords)*

---

**Passo 1 — Vocabulário**

$$V = \{\text{i},\ \text{love},\ \text{you}\}$$

---

**Passo 2 — Matriz BoW (contagens)**

Linhas = termos, colunas = documentos $\{d_1, d_2\}$, onde $d_1 =$ "i love you" e $d_2 =$ "love":

$$
\text{BoW} =
\begin{array}{c|cc}
 & d_1 & d_2 \\
\hline
\text{i} & 1 & 0 \\
\text{love} & 1 & 1 \\
\text{you} & 1 & 0
\end{array}
$$

Em notação matricial pura:

$$
\text{BoW} =
\begin{bmatrix}
1 & 0 \\
1 & 1 \\
1 & 0
\end{bmatrix}
$$

---

**Passo 3 — Document Frequency ($n_t$)**

$$n_{\text{i}} = 1, \qquad n_{\text{love}} = 2, \qquad n_{\text{you}} = 1$$

---

**Passo 4 — IDF suavizado**

$$\text{IDF}(t) = \ln\!\left(\frac{N+1}{n_t+1}\right) + 1, \qquad N = 2$$

$$\text{IDF}(\text{i}) = \ln\!\left(\frac{2+1}{1+1}\right) + 1 = \ln\!\left(\frac{3}{2}\right) + 1 \approx 0.405465 + 1 = 1.405465$$

$$\text{IDF}(\text{love}) = \ln\!\left(\frac{2+1}{2+1}\right) + 1 = \ln(1) + 1 = 0 + 1 = 1$$

$$\text{IDF}(\text{you}) = \ln\!\left(\frac{2+1}{1+1}\right) + 1 = \ln\!\left(\frac{3}{2}\right) + 1 \approx 0.405465 + 1 = 1.405465$$

---

**Passo 5 — Matriz diagonal IDF**

$$
\Sigma_{\text{IDF}} =
\begin{bmatrix}
1.405465 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1.405465
\end{bmatrix}
$$

(ordem das linhas/colunas segue $V = \{\text{i}, \text{love}, \text{you}\}$)

---

**Passo 6 — TF-IDF bruto**

Cada célula de $\text{BoW}$ é multiplicada pelo IDF do seu termo (linha), ou seja $\text{TFIDF}_{raw} = \Sigma_{\text{IDF}} \cdot \text{BoW}$:

$$
\Sigma_{\text{IDF}} \cdot \text{BoW} =
\begin{bmatrix}
1.405465 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1.405465
\end{bmatrix}
\begin{bmatrix}
1 & 0 \\
1 & 1 \\
1 & 0
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
1.405465 \times 1 & 1.405465 \times 0 \\
1 \times 1 & 1 \times 1 \\
1.405465 \times 1 & 1.405465 \times 0
\end{bmatrix}
=
\begin{bmatrix}
1.405465 & 0 \\
1 & 1 \\
1.405465 & 0
\end{bmatrix}
$$

Ou seja:

$$
\text{TFIDF}_{raw} =
\begin{array}{c|cc}
 & d_1 & d_2 \\
\hline
\text{i} & 1.405465 & 0 \\
\text{love} & 1 & 1 \\
\text{you} & 1.405465 & 0
\end{array}
$$

---

**Passo 7 — Normalização L2 por documento (coluna)**

$$\|\mathbf{d_1}\|_2 = \sqrt{1.405465^2 + 1^2 + 1.405465^2} = \sqrt{1.975334 + 1 + 1.975334} = \sqrt{4.950668} \approx 2.224999$$

$$\|\mathbf{d_2}\|_2 = \sqrt{0^2 + 1^2 + 0^2} = \sqrt{1} = 1$$

---

**Passo 8 — Matriz TF-IDF final (normalizada)**

$$
d_1^{norm} =
\left[
\frac{1.405465}{2.224999},\ \ \frac{1}{2.224999},\ \ \frac{1.405465}{2.224999}
\right]
\approx
\left[0.631668,\ \ 0.449436,\ \ 0.631668\right]
$$

$$
d_2^{norm} =
\left[\frac{0}{1},\ \ \frac{1}{1},\ \ \frac{0}{1}\right]
=
\left[0,\ \ 1,\ \ 0\right]
$$

$$
\text{TFIDF}_{final} =
\begin{array}{c|cc}
 & d_1 \text{ ("i love you")} & d_2 \text{ ("love")} \\
\hline
\text{i} & 0.631668 & 0 \\
\text{love} & 0.449436 & 1 \\
\text{you} & 0.631668 & 0
\end{array}
$$

Em notação matricial pura:

$$
\text{TFIDF}_{final} =
\begin{bmatrix}
0.631668 & 0 \\
0.449436 & 1 \\
0.631668 & 0
\end{bmatrix}
$$

Essa matriz final corresponde exatamente à saída do R (`resultado_tfidf$matrix`), com termos nas linhas e documentos nas colunas.

#### **Sub-teste: remoção de stopwords ativada:**

In [13]:
tokens <- get_processed_tokens(
    textos_simples, 
    dontRemoveStopWords = FALSE)
print(tokens)

[[1]]
[1] "love"

[[2]]
[1] "love"



In [14]:
resultado_bow <- get_BOW_matrix(
    textos_simples, 
    dontRemoveStopWords = FALSE)
    
resultado_bow$matrix

,love,love
love,1,1


In [15]:
resultado_tfidf <- get_TFIDF_matrix(
    textos_simples, 
    dontRemoveStopWords = FALSE)

resultado_tfidf$matrix

,love,love
love,1,1


### **Parte 3 - Módulo feito para o teste da similaridade do cosseno:**

In [16]:
cosine_similarity_native <- function(m) {
  # m: matriz termos x documentos (colunas = documentos),
  # que é o formato retornado por get_TFIDF_matrix.
  norms <- sqrt(colSums(m^2))
  norms[norms == 0] <- 1
  m_normalized <- sweep(m, 2, norms, "/")

  # produto interno entre documentos normalizados = cosseno
  sim <- t(m_normalized) %*% m_normalized
  return(sim)
}

getCossine_Similarity_matrix <- function(nonProcessedTexts = list(),
                                          tf_idf_matrix = NULL,
                                          cancelStopWordRemoval = FALSE) {

  if (length(nonProcessedTexts) > 0 && is.null(tf_idf_matrix)) {

    resultado_tfidf <- get_TFIDF_matrix(nonProcessedTexts, cancelStopWordRemoval)
    tf_idf_matrix <- resultado_tfidf$matrix

    similarity <- cosine_similarity_native(tf_idf_matrix)
    return(similarity)

  } else if (!is.null(tf_idf_matrix)) {

    similarity <- cosine_similarity_native(tf_idf_matrix)

    if (cancelStopWordRemoval == TRUE) {
      cat(">>> AVISO: Matriz TF-IDF utilizada como parâmetro: <<<\n")
      cat("-> Comando digitado: getCossine_Similarity_matrix(NULL, tf_idf_matrix, TRUE)\n")
      cat("-> IMPOSSÍVEL desligar remoção de STOP-WORDS pré estabelecida...\n")
      cat("-> SOLUÇÃO: getCossine_Similarity_matrix(nonProcessedTexts, NULL, TRUE)\n\n")
    }

    return(similarity)
  }

  return(NULL)
}

In [17]:
textos <- list(
    "Amo comer pizza.",
    "Amo comer hamburguer.",
    "Amo lutar, mano.",
    "Odeio lutar",
    "Lutar é bom!",
    "Um gato pulou da janela..."
)

In [18]:
# Obtendo Matriz de Similaridade:

sim_1 <- getCossine_Similarity_matrix(
  nonProcessedTexts = textos,
  tf_idf_matrix = NULL,
  cancelStopWordRemoval = TRUE # Remoção de stopwords
)                              # desativada!
print(sim_1)

                        amo comer pizza amo comer hamburguer amo lutar mano
amo comer pizza               1.0000000            0.5352555      0.2334744
amo comer hamburguer          0.5352555            1.0000000      0.2334744
amo lutar mano                0.2334744            0.2334744      1.0000000
odeio lutar                   0.0000000            0.0000000      0.2815818
lutar é bom                   0.0000000            0.0000000      0.2175043
um gato pulou da janela       0.0000000            0.0000000      0.0000000
                        odeio lutar lutar é bom um gato pulou da janela
amo comer pizza           0.0000000   0.0000000                       0
amo comer hamburguer      0.0000000   0.0000000                       0
amo lutar mano            0.2815818   0.2175043                       0
odeio lutar               1.0000000   0.2502721                       0
lutar é bom               0.2502721   1.0000000                       0
um gato pulou da janela   0.0000000 

In [19]:
# Obtendo Matriz de Similaridade:

sim_1 <- getCossine_Similarity_matrix(
  nonProcessedTexts = textos,
  tf_idf_matrix = NULL,
  cancelStopWordRemoval = FALSE # Remoção de stopwords
)                               # ativada!
print(sim_1)

                     amo comer pizza amo comer hamburguer amo lutar mano
amo comer pizza            1.0000000            0.5352555      0.2334744
amo comer hamburguer       0.5352555            1.0000000      0.2334744
amo lutar mano             0.2334744            0.2334744      1.0000000
odeio lutar                0.0000000            0.0000000      0.2815818
lutar é bom                0.0000000            0.0000000      0.2175043
gato pulou janela          0.0000000            0.0000000      0.0000000
                     odeio lutar lutar é bom gato pulou janela
amo comer pizza        0.0000000   0.0000000                 0
amo comer hamburguer   0.0000000   0.0000000                 0
amo lutar mano         0.2815818   0.2175043                 0
odeio lutar            1.0000000   0.2502721                 0
lutar é bom            0.2502721   1.0000000                 0
gato pulou janela      0.0000000   0.0000000                 1


### **Parte 4A - Módulo responsável por achar os N termos mais próximos do que foi digitado:**

In [20]:
# Os códigos abaixo giram em torno de recomendar,
# a partir de uma consulta (query) do usuário, os
# 'top_n' documentos mais próximos com base na
# similaridade do cosseno sobre a matriz TF-IDF.
#
# A fonte real dos documentos (arquivo, banco, etc.)
# ainda não existe -- por isso "lista_documentos" entra
# como argumento, para ser preenchido depois por quem
# for buscar os dados de verdade.

recomendar_documentos <- function(query,
                                  lista_documentos,
                                  top_n = 3,
                                  dontRemoveStopWords = FALSE) {

  # query entra como o "documento 0" do corpus:
  textos <- c(list(query), lista_documentos)

  resultado_tfidf <- get_TFIDF_matrix(textos, dontRemoveStopWords)
  tfidf_matrix <- resultado_tfidf$matrix  # termos x documentos (query = coluna 1)

  similarity_matrix <- getCossine_Similarity_matrix(
    tf_idf_matrix = tfidf_matrix)
  
  # similarity_matrix é documentos x documentos, na mesma ordem das colunas
  # de tfidf_matrix -- ou seja, linha/coluna 1 = query.

  similarities <- similarity_matrix[1, -1]  # query vs. cada documento (exclui a própria query)

  n_disponiveis <- min(top_n, length(similarities))
  indices <- order(similarities, decreasing = TRUE)[1:n_disponiveis]

  resultados <- lapply(indices, function(idx) {
    list(documento = lista_documentos[[idx]],
         similaridade = similarities[idx])
  })

  return(resultados)
}

executar_recomendacao_ao_usuario <- function(fonteDocumentos,
                                              queryEscritaPeloUsuario,
                                              top_n = 3) {

  cat("\n=== BUSCAR DOCUMENTOS MAIS PRÓXIMOS DA CONSULTA: ===\n")
  cat(sprintf("\nConsulta: \"%s\"\n", queryEscritaPeloUsuario))

  resultados <- recomendar_documentos(
    query = queryEscritaPeloUsuario,
    lista_documentos = fonteDocumentos,
    top_n = top_n
  )

  for (i in seq_along(resultados)) {
    cat(sprintf("   %d. Score: %.3f\n", i, resultados[[i]]$similaridade))
    cat(sprintf("   Documento: %s\n", resultados[[i]]$documento))
  }

  return(invisible(resultados))
}

### **Parte 4B - Simulação da busca de um usuário:**

In [21]:
# Exemplo de uso: recomendação de documentos mais próximos
# de uma consulta pessoal do usuário.

textos <- list(
  "bolo de chocolate",
  "bolo de nozes e chocolate",
  "café com chocolate",
  "Ovomaltin com café",
  "cookies 'n cream e bolo de chocolate, receita fácil",
  "Hamburguer vegano com ovos e bacon"
)

queryEscritaPeloUsuario <- "Receita de bolo de chocolate com cookies'n cream"

resultados <- executar_recomendacao_ao_usuario(
  fonteDocumentos = textos,
  queryEscritaPeloUsuario = queryEscritaPeloUsuario,
  top_n = 3
)


=== BUSCAR DOCUMENTOS MAIS PRÓXIMOS DA CONSULTA: ===

Consulta: "Receita de bolo de chocolate com cookies'n cream"
   1. Score: 0.522
   Documento: cookies 'n cream e bolo de chocolate, receita fácil
   2. Score: 0.469
   Documento: bolo de chocolate
   3. Score: 0.297
   Documento: bolo de nozes e chocolate


### **Parte 4C - Simulação de busca usando o *corpus* da AULA 01:**

In [22]:
docs <- c(
    d1 = "recuperacao de informacao ordena documentos por relevancia",
    d2 = "o modelo de espaco vetorial representa documentos como vetores",
    d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",
    d4 = "aprendizado estatistico fundamenta a recuperacao moderna",
    d5 = "o indice invertido acelera a busca em muitos documentos",
    d6 = "embeddings capturam a semantica de palavras e documentos",
    d7 = "a avaliacao mede a relevancia dos resultados da busca",
    d8 = "ciencia de dados combina estatistica e programacao"
)

In [23]:
queryEscritaPeloUsuario <- "modelo de espaco vetorial para documentos."

resultados <- executar_recomendacao_ao_usuario(
  fonteDocumentos = docs,
  queryEscritaPeloUsuario = queryEscritaPeloUsuario,
  top_n = 3
)


=== BUSCAR DOCUMENTOS MAIS PRÓXIMOS DA CONSULTA: ===

Consulta: "modelo de espaco vetorial para documentos."
   1. Score: 0.731
   Documento: o modelo de espaco vetorial representa documentos como vetores
   2. Score: 0.167
   Documento: bm25 e um modelo probabilistico de ranqueamento de texto
   3. Score: 0.114
   Documento: recuperacao de informacao ordena documentos por relevancia


### **Parte 5A - Configurando Web Scrapping via R:**

O scrapping abaixo pega do jornal A Tribuna das cidades de Guarujá, Santos e Bertioga.

In [25]:
# Load required libraries
if (!require(rvest)) install.packages("rvest")
if (!require(pacman)) install.packages("pacman")
library(rvest)
library(pacman)

# City configuration
cities <- list(
  guaruja = list(
    name = "Guarujá",
    cid = "1.499556",
    base_url = "https://www.atribuna.com.br/buscar?page=%d&cid=%s&pageStart=%d"
  ),
  santos = list(
    name = "Santos",
    cid = "1.499607",
    base_url = "https://www.atribuna.com.br/buscar?page=%d&cid=%s&pageStart=%d"
  ),
  bertioga = list(
    name = "Bertioga",
    cid = "1.499531",
    base_url = "https://www.atribuna.com.br/buscar?page=%d&cid=%s&pageStart=%d"
  )
)

# ============================================
# NEW: Random delay function with your logic
# ============================================

random_delay <- function() {
  # Generate a random key between 0 and 20
  key <- runif(1, 0, 20)
  
  if (key <= 10) {
    delay <- 3
    cat(sprintf("  [Key: %.2f] Waiting %d seconds...\n", key, delay))
  } else {
    delay <- 6
    cat(sprintf("  [Key: %.2f] Waiting %d seconds...\n", key, delay))
  }
  
  Sys.sleep(delay)
}

# OR, if you want more variation (recommended):
random_delay_varied <- function() {
  # Generate a random key between 0 and 20
  key <- runif(1, 0, 20)
  
  if (key <= 10) {
    # 3-4 seconds variation
    delay <- runif(1, 3, 4)
  } else {
    # 6-8 seconds variation
    delay <- runif(1, 6, 8)
  }
  
  cat(sprintf("  [Key: %.2f] Waiting %.1f seconds...\n", key, delay))
  Sys.sleep(delay)
}

# ============================================
# UPDATED: scrape_city_news WITH RANDOM DELAYS
# ============================================

scrape_city_news <- function(city_key, max_articles = 100, use_varied = TRUE) {
  city <- cities[[city_key]]
  if (is.null(city)) {
    stop("City not found. Available: guaruja, santos, bertioga")
  }
  
  cat(sprintf("\n=== Scraping news from %s ===\n", city$name))
  
  all_news <- data.frame()
  page_num <- 1
  page_start <- 0
  base_url <- city$base_url
  cid <- city$cid
  page_count <- 0
  
  while (nrow(all_news) < max_articles) {
    url <- sprintf(base_url, page_num, cid, page_start)
    page_count <- page_count + 1
    cat(sprintf("\nPage %d: %s\n", page_num, url))
    
    # Apply random delay BEFORE each request (except first page)
    if (page_count > 1) {
      if (use_varied) {
        random_delay_varied()
      } else {
        random_delay()
      }
    }
    
    pagina <- tryCatch(read_html(url), error = function(e) NULL)
    if (is.null(pagina)) {
      cat("Failed to load page. Stopping.\n")
      break
    }
    
    teasers <- c(
      pagina %>% html_elements(".Teaser.mobile"),
      pagina %>% html_elements(".Teaser.desktop")
    )
    
    if (length(teasers) == 0) {
      cat("No more articles found. Stopping.\n")
      break
    }
    
    results <- list()
    for (teaser in teasers) {
      title_link <- teaser %>% html_element(".TeaserTitleText a")
      if (!is.na(title_link)) {
        results[[length(results) + 1]] <- data.frame(
          titulo = title_link %>% html_text(trim = TRUE),
          url = paste0("https://www.atribuna.com.br", title_link %>% html_attr("href")),
          categoria = teaser %>% html_element(".TeaserSubjectText") %>% html_text(trim = TRUE),
          stringsAsFactors = FALSE
        )
      }
    }
    
    if (length(results) == 0) {
      cat("No articles on this page. Stopping.\n")
      break
    }
    
    df_page <- do.call(rbind, results)
    all_news <- rbind(all_news, df_page)
    all_news <- all_news[!duplicated(all_news$url), ]
    
    cat(sprintf("  Total articles so far: %d\n", nrow(all_news)))
    
    page_num <- page_num + 1
    page_start <- page_start + 10
  }
  
  cat(sprintf("\n=== Finished scraping %s ===\n", city$name))
  cat(sprintf("Total articles collected: %d\n", nrow(all_news)))
  
  return(all_news)
}

# ============================================
# UPDATED: search_city_news (with output_dir parameter)
# ============================================

search_city_news <- function(city_key, query, top_n = 3, max_articles = 100, 
                             use_varied = TRUE, output_dir = ".") {
  # Step 1: Scrape news from the city with random delays
  df_news <- scrape_city_news(city_key, max_articles, use_varied)
  
  if (nrow(df_news) == 0) {
    cat("No articles found for this city.\n")
    return(NULL)
  }
  
  # Step 2: Save the dataframe (side effect) in the specified folder
  city_name <- cities[[city_key]]$name
  
  # Create output directory if it doesn't exist
  if (!dir.exists(output_dir)) {
    dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
    cat(sprintf("Created directory: %s\n", output_dir))
  }
  
  filename <- file.path(output_dir, paste0("noticias_", tolower(city_name), ".csv"))
  write.csv(df_news, filename, row.names = FALSE, fileEncoding = "UTF-8")
  cat(sprintf("\nDataframe saved as: %s\n", filename))
  
  # Step 3: Use your existing function to find similar articles
  fonteDocumentos <- as.list(df_news$titulo)
  
  cat(sprintf("\n=== Searching for: \"%s\" in %s news ===\n", query, city_name))
  
  resultados <- executar_recomendacao_ao_usuario(
    fonteDocumentos = fonteDocumentos,
    queryEscritaPeloUsuario = query,
    top_n = top_n
  )
  
  return(invisible(list(
    city = city_name,
    query = query,
    results = resultados,
    dataframe = df_news
  )))
}

# ============================================
# INTERACTIVE VERSION (with output_dir prompt)
# ============================================

interactive_search <- function() {
  cat("\n=== CITY NEWS SEARCH ===\n")
  cat("Available cities:\n")
  cat("1. Guarujá\n")
  cat("2. Santos\n")
  cat("3. Bertioga\n")
  
  city_choice <- as.numeric(readline("Choose a city (1, 2, or 3): "))
  city_keys <- c("guaruja", "santos", "bertioga")
  city_key <- city_keys[city_choice]
  
  if (is.na(city_key)) {
    cat("Invalid choice.\n")
    return(NULL)
  }
  
  query <- readline("Enter your search phrase: ")
  
  if (query == "") {
    cat("No query entered.\n")
    return(NULL)
  }
  
  top_n <- as.numeric(readline("How many results to show? (default 3): "))
  if (is.na(top_n) || top_n < 1) top_n <- 3
  
  max_articles <- as.numeric(readline("Max articles to scrape? (default 30): "))
  if (is.na(max_articles) || max_articles < 1) max_articles <- 30
  
  # NEW: Ask for output folder
  output_dir <- readline("Folder to save CSV (press Enter for current directory): ")
  if (output_dir == "") output_dir <- "."
  
  search_city_news(city_key, query, top_n, max_articles, 
                   use_varied = TRUE, output_dir = output_dir)
}

# ============================================
# EXAMPLE USAGE:
# ============================================

# Example 1: Search in Guarujá news (with random delays)
# Save files in a folder called "meus_dados"
# resultados_guaruja <- search_city_news(
#   city_key = "guaruja",
#   query = "Guarujá cria",
#   top_n = 5,
#   max_articles = 30,
#   use_varied = TRUE,
#   output_dir = "./meus_dados"
# )

# Example 2: Quick interactive version
# interactive_search()

Carregando pacotes exigidos: rvest



In [26]:
URL_DOWNLOADS <- r"(C:\Users\Ivan\Documents\Pasta-Documentos-PC-antigo\GITHUB-Meus-Repositorios\PesquisaPI3_2026\data\raw)"

#### **Parte 6A - Busca e ranqueamento de notícias (A Tribuna - "Guarujá"):**

In [ ]:
resultados_guaruja <- search_city_news(
  city_key = "guaruja",
  query = "Guarujá cria",
  top_n = 5,
  max_articles = 20,
  use_varied = TRUE,
  output_dir = URL_DOWNLOADS
)


=== Scraping news from Guarujá ===

Page 1: https://www.atribuna.com.br/buscar?page=1&cid=1.499556&pageStart=0
  Total articles so far: 10

Page 2: https://www.atribuna.com.br/buscar?page=2&cid=1.499556&pageStart=10
  [Key: 16.13] Waiting 7.4 seconds...
  Total articles so far: 20

=== Finished scraping Guarujá ===
Total articles collected: 20

Dataframe saved as: ./noticias_guarujá.csv

=== Searching for: "Guarujá cria" in Guarujá news ===

=== BUSCAR DOCUMENTOS MAIS PRÓXIMOS DA CONSULTA: ===

Consulta: "Guarujá cria"
   1. Score: 0.378
   Documento: Guarujá cria comitê para enfrentar crise provocada por falta de água
   2. Score: 0.051
   Documento: Fatec terá aulas presenciais a partir de 2027 em Guarujá, no litoral de São Paulo
   3. Score: 0.050
   Documento: Museu de Guarujá reabre com acervo que resgata a história da Cidade
   4. Score: 0.049
   Documento: Guarujá, no litoral de São Paulo, libera agendamento de consultas pelo celular; veja como usar
   5. Score: 0.047
   Docume

#### **Parte 6B - Busca e ranqueamento de notícias (A Tribuna - "Santos"):**

In [31]:
resultados_guaruja <- search_city_news(
  city_key = "santos",
  query = "Propaganda Eleitoral é liberada",
  top_n = 5,
  max_articles = 20,
  use_varied = TRUE,
  output_dir = URL_DOWNLOADS
)


=== Scraping news from Santos ===

Page 1: https://www.atribuna.com.br/buscar?page=1&cid=1.499607&pageStart=0
  Total articles so far: 10

Page 2: https://www.atribuna.com.br/buscar?page=2&cid=1.499607&pageStart=10
  [Key: 6.64] Waiting 3.1 seconds...
  Total articles so far: 20

=== Finished scraping Santos ===
Total articles collected: 20

Dataframe saved as: C:\Users\Ivan\Documents\Pasta-Documentos-PC-antigo\GITHUB-Meus-Repositorios\PesquisaPI3_2026\data\raw/noticias_santos.csv

=== Searching for: "Propaganda Eleitoral é liberada" in Santos news ===

=== BUSCAR DOCUMENTOS MAIS PRÓXIMOS DA CONSULTA: ===

Consulta: "Propaganda Eleitoral é liberada"
   1. Score: 0.121
   Documento: Ela passou mal no vestibular de Medicina; 10 anos depois, é médica em Santos
   2. Score: 0.097
   Documento: André Mendonça, ministro do STF que tirou sigilo de mensagens entre Alexandre de Moraes e Vorcaro, é de Santos; conheça
   3. Score: 0.000
   Documento: CVV busca voluntários em Santos para ampliar 

#### **Parte 6C - Busca e ranqueamento de notícias (A Tribuna - "Bertioga"):**

In [32]:
resultados_guaruja <- search_city_news(
  city_key = "bertioga",
  query = "Bertioga completa 35 anos",
  top_n = 5,
  max_articles = 20,
  use_varied = TRUE,
  output_dir = URL_DOWNLOADS
)


=== Scraping news from Bertioga ===

Page 1: https://www.atribuna.com.br/buscar?page=1&cid=1.499531&pageStart=0
  Total articles so far: 10

Page 2: https://www.atribuna.com.br/buscar?page=2&cid=1.499531&pageStart=10
  [Key: 16.93] Waiting 6.8 seconds...
  Total articles so far: 20

=== Finished scraping Bertioga ===
Total articles collected: 20

Dataframe saved as: C:\Users\Ivan\Documents\Pasta-Documentos-PC-antigo\GITHUB-Meus-Repositorios\PesquisaPI3_2026\data\raw/noticias_bertioga.csv

=== Searching for: "Bertioga completa 35 anos" in Bertioga news ===

=== BUSCAR DOCUMENTOS MAIS PRÓXIMOS DA CONSULTA: ===

Consulta: "Bertioga completa 35 anos"
   1. Score: 0.674
   Documento: Bertioga completa 35 anos de emancipação e traça metas para os próximos anos no litoral de São Paulo
   2. Score: 0.420
   Documento: Bertioga aposta em ecoturismo e crescimento ordenado ao completar 35 anos no litoral de São Paulo
   3. Score: 0.114
   Documento: Homem que acompanhava prova de caiaque desmaia